# Notebook 04 — Results and Figures

**Question:** what does the detector produce for each of the three runs, and which single
figure shows it?

Generates the per-test summary dashboards and one comparison figure across all three runs,
then writes a raw detection summary.

The lead-time and downtime-cost analysis is deliberately **not** here. It needs a
sustained-alert rule and a false-alarm rate calibrated on a healthy baseline, which
`business.py` adds. The intervals this notebook prints are raw first-alert numbers and are
labelled as such.

---

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from nasa_bearing_anomaly.config import TEST_CONFIG
from nasa_bearing_anomaly.detection import run_pipeline, select_features
from nasa_bearing_anomaly.plotting import BearingPlotter

FIGURES_DIR = Path("../results/figures")
REPORTS_DIR = Path("../results/reports")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Ready")

## 1. Load Results (or Re-run if Needed)

In [ ]:
results = {}
for test_id in [1, 2, 3]:
    result_path = Path(f"../results/reports/test{test_id}_isolation_forest_results.csv")
    if result_path.exists():
        results[test_id] = pd.read_csv(result_path, index_col="file_index")
        print(f"✅ Test {test_id}: Loaded from {result_path}")
    else:
        print(f"⏳ Test {test_id}: Running pipeline...")
        results[test_id] = run_pipeline(test_id)

## 2. Summary Dashboard (Per Test)

The main portfolio image — one per test.

In [ ]:
for test_id, df in results.items():
    print(f"\n📊 Generating dashboard for Test {test_id}...")
    plotter = BearingPlotter(test_id=test_id, save_figures=True)
    config = TEST_CONFIG[test_id]
    feature_cols = select_features(df, bearing_prefix=config["failed_bearing"])
    fig = plotter.plot_summary_dashboard(df, feature_cols=feature_cols)
    plt.show()

## 3. All-Tests Comparison Figure

One clean figure showing all 3 tests side by side — for README header.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 13))
fig.patch.set_facecolor("#0d1117")
fig.suptitle(
    "NASA IMS Bearing Dataset — Predictive Maintenance via Anomaly Detection\n"
    "All Three Test Runs | Isolation Forest | Industry 4.0",
    fontsize=13,
    fontweight="bold",
    color="#e6edf3",
    y=0.99,
)

bearing_colors = ["#58a6ff", "#3fb950", "#d29922", "#bc8cff"]

for row, (test_id, df) in enumerate(results.items()):
    config = TEST_CONFIG[test_id]
    failed = config["failed_bearing"]
    rms_col = f"{failed}_ch1_rms"

    ax_rms = axes[row, 0]
    ax_score = axes[row, 1]

    for ax in [ax_rms, ax_score]:
        ax.set_facecolor("#161b22")

    # ── Left: All bearings RMS ──
    for i, bearing in enumerate(["Bearing1", "Bearing2", "Bearing3", "Bearing4"]):
        col = f"{bearing}_ch1_rms"
        if col in df.columns:
            lw = 1.5 if bearing == failed else 0.6
            alpha = 1.0 if bearing == failed else 0.5
            lbl = bearing + (" (failed)" if bearing == failed else "")
            ax_rms.plot(
                df.index, df[col], color=bearing_colors[i], linewidth=lw, alpha=alpha, label=lbl
            )

    ax_rms.set_title(
        f"Test {test_id}: RMS — {config['failure_mode']}",
        fontsize=10,
        color="#e6edf3",
        fontweight="bold",
    )
    ax_rms.set_ylabel("RMS (g)", color="#e6edf3")
    ax_rms.legend(fontsize=7, loc="upper left")
    ax_rms.grid(True, color="#30363d", alpha=0.6)
    ax_rms.tick_params(colors="#e6edf3")

    # ── Right: Anomaly detection result ──
    if rms_col in df.columns:
        ax_score.plot(df.index, df[rms_col], color="#58a6ff", linewidth=0.8, alpha=0.7, label="RMS")

        if "is_anomaly" in df.columns:
            anom_mask = df["is_anomaly"]
            ax_score.scatter(
                df.index[anom_mask],
                df[rms_col][anom_mask],
                color="#f85149",
                s=6,
                zorder=5,
                label="Anomaly",
            )

            first_anom = df.index[anom_mask].min() if anom_mask.any() else len(df)
            ax_score.axvline(
                x=first_anom,
                color="#d29922",
                linestyle="--",
                linewidth=1.5,
                label=f"Alert at #{first_anom}",
            )
            ax_score.text(
                first_anom + max(2, len(df) * 0.01),
                ax_score.get_ylim()[1] * 0.75 if ax_score.get_ylim()[1] > 0 else 0.5,
                f"First alert\nfile {first_anom} of {len(df)}",
                color="#d29922",
                fontsize=7,
            )

    ax_score.set_title(
        f"Test {test_id}: Anomaly Detection Result", fontsize=10, color="#e6edf3", fontweight="bold"
    )
    ax_score.set_ylabel(f"{failed} RMS (g)", color="#e6edf3")
    ax_score.legend(fontsize=7, loc="upper left")
    ax_score.grid(True, color="#30363d", alpha=0.6)
    ax_score.tick_params(colors="#e6edf3")

for ax in axes[-1]:
    ax.set_xlabel("File Index (Time →)", color="#e6edf3")

plt.tight_layout()
out_path = FIGURES_DIR / "all_tests_comparison.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="#0d1117")
print(f"✅ Saved: {out_path}")
plt.show()

## 4. Business Impact Report

In [ ]:
# Raw detection summary. This is NOT the business report -- see the caveats printed
# below, and business.py, which is where lead time becomes a result.
report_rows = []
for test_id, df in results.items():
    config = TEST_CONFIG[test_id]
    n = len(df)
    n_anom = df["is_anomaly"].sum() if "is_anomaly" in df.columns else 0
    first_anom = df[df["is_anomaly"]].index.min() if n_anom > 0 else n

    # Measured from the acquisition timestamps rather than an assumed interval.
    stamps = pd.to_datetime(df["timestamp"])
    raw_lead_h = (
        (stamps.iloc[-1] - stamps.loc[first_anom]).total_seconds() / 3600 if n_anom > 0 else 0.0
    )

    report_rows.append(
        {
            "test": test_id,
            "failed_bearing": config["failed_bearing"],
            "failure_mode": config["failure_mode"],
            "files": n,
            "anomalies_flagged": n_anom,
            "first_alert_file": first_anom,
            "raw_lead_hours": round(raw_lead_h, 1),
        }
    )

report_df = pd.DataFrame(report_rows)
report_df.to_csv(REPORTS_DIR / "detection_summary.csv", index=False)

print("DETECTION SUMMARY")
print("=" * 78)
print(report_df.to_string(index=False))
print("=" * 78)
print()
print("raw_lead_hours is the interval from the first flagged file to the last file of the")
print("run. It is not a lead-time result, and no euro figure follows from it:")
print("  - it triggers on the FIRST anomaly, so a single early false positive inflates it")
print("  - it carries no false-alarm rate, so there is nothing to weigh it against")
print("  - the last file of each run is post-shutdown, so it is not the failure moment")
print()
print("business.py adds the sustained-alert rule, the false-alarm rate and the")
print("downtime-cost model. Saved: results/reports/detection_summary.csv")

## 5. Final: Copy Best Figure to README Location

The `all_tests_comparison.png` is the main image for the GitHub README.

In [ ]:
import shutil

src = FIGURES_DIR / "all_tests_comparison.png"
dst = FIGURES_DIR / "README_main_figure.png"
if src.exists():
    shutil.copy(src, dst)
    print(f"Saved: {dst}")
    print("   Embed in README.md as: ![Results](results/figures/README_main_figure.png)")